# RSICCformer: Colab training and evaluation

Select a GPU runtime, edit the configuration cell, then run all cells in order.

Each run gets its own folder in Google Drive with `run.json` (code commit, configuration, dataset hash, seeds, environment and results), logs, checkpoints and test results. See the README for details.

## 1. Configuration

In [ ]:
# Code version: branch, tag or commit. Use a tag or commit for results you will report.
REPO_URL = "https://github.com/yakupgulcan/RSICC.git"
REPO_REF = "thesis"
OUTPUT_ROOT = "/content/drive/MyDrive/RSICC_Outputs"  # must be under MyDrive

# Data
EXPERIMENT = "baseline"  # short name; becomes part of the run folder name
DATASET_SOURCE = "https://drive.google.com/file/d/1YgWES-0OOL-3KK4yIV3-y8MN-_c1ELEE/view?usp=drive_link"  # ZIP (URL or Drive path) with LevirCCcaptions.json and images/
PREPROCESSED_SOURCE = None  # optional ZIP of ready HDF5/JSON training files; replaces DATASET_SOURCE
DATA_SEED = 123  # caption sampling when training files are built; keep fixed across runs
SAVE_PREPROCESSED = False  # also save the built training files to the run folder

# Training
SEED = 123  # None: choose a random seed and record it
DETERMINISTIC = False  # True: deterministic PyTorch algorithms (slower; unsupported operations raise)
TOTAL_EPOCHS = 20  # total target, including completed epochs when resuming
BATCH_SIZE = 32
NUM_WORKERS = 4
STOP_CRITERIA = 5  # epochs without validation BLEU-4 improvement before stopping
ENCODER_LR = 1e-4
DECODER_LR = 1e-4
HARMONIC_ALPHA = 0.0  # harmonic smoothing of the CNN features on a shared graph; 0 disables it
EXTRA_TRAIN_ARGS = []  # further train.py arguments, e.g. ["--n_layers", "2"]
RESUME_CHECKPOINT = None  # Drive path of a checkpoint_*.pth.tar to continue from

# Evaluation
EVAL_CHECKPOINT = None  # None: best checkpoint of this run; otherwise a checkpoint path
EVAL_BEAM_SIZE = 1
DISCONNECT_WHEN_DONE = True  # release the runtime after the last cell

# Saved in run.json. Only plain settings are kept, so re-running this cell later stays harmless.
CONFIG = {name: value for name, value in dict(globals()).items()
          if name.isupper() and isinstance(value, (str, int, float, bool, list, type(None)))}

## 2. Setup

Mounts Drive, clones the code and installs dependencies. Use a fresh runtime for each run.

In [ ]:
import os, subprocess, sys
from pathlib import Path
from google.colab import auth, drive

drive.mount("/content/drive")
auth.authenticate_user()  # Drive API access for checkpoint uploads

PROJECT_DIR = Path("/content/RSICC")
if PROJECT_DIR.exists():
    raise FileExistsError(f"{PROJECT_DIR} already exists. Use a fresh runtime.")
subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=PROJECT_DIR, check=True)
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
%pip install --quiet -r requirements.txt gdown google-api-python-client google-auth

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab.")
subprocess.run(["java", "-version"], check=True)  # METEOR needs Java

## 3. Start the run

Creates the run folder and `run.json`.

In [ ]:
import json, shutil, tempfile
from datetime import datetime, timezone
from drive_checkpoints import DriveCheckpoints, checksum, drive_service, resolve_folder
from notebook_runtime import (extract_raw_dataset, fetch_archive, required_data_files,
                              restore_preprocessed_data, run_logged, start_run,
                              training_summary, update_run)
from reproducibility import resolve_seed


def timestamp():
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")


RUN_SEED = resolve_seed(SEED)
RUN_ID = f"{EXPERIMENT}_seed{RUN_SEED}_{timestamp()}"
RUN_DIR = Path(OUTPUT_ROOT) / "runs" / RUN_ID  # Drive: records, logs, checkpoints, results
WORK_DIR = PROJECT_DIR / "runs" / RUN_ID  # runtime storage: data and local checkpoints
DATA_DIR, CHECKPOINT_DIR = WORK_DIR / "data", WORK_DIR / "models_checkpoint"
DATA_NAME = "LEVIR_CC_5_cap_per_img_5_min_word_freq"
CHECKPOINT_NAME = "checkpoint_resnet101_MCCFormers_diff_as_Q_trans.pth.tar"
SEED_ARGS = ["--seed", str(RUN_SEED)] + (["--deterministic"] if DETERMINISTIC else [])
os.environ["PYTHONHASHSEED"] = str(RUN_SEED)

start_run(RUN_DIR, {**CONFIG, "SEED": RUN_SEED}, PROJECT_DIR)
DATA_DIR.mkdir(parents=True)
CHECKPOINT_DIR.mkdir()
(RUN_DIR / "models_checkpoint").mkdir()
service = drive_service()
# The folder was just created through the mount; wait until Drive itself lists it.
DRIVE_CHECKPOINT_FOLDER_ID = resolve_folder(service, RUN_DIR / "models_checkpoint", wait_seconds=300)
print((RUN_DIR / "run.json").read_text())

## 4. Prepare the data

Downloads the dataset and builds the training files, or restores ready training files when `PREPROCESSED_SOURCE` is set.

In [ ]:
source = PREPROCESSED_SOURCE or DATASET_SOURCE
archive, archive_sha256 = fetch_archive(source, "/content/dataset.zip", RUN_DIR / "data.log")
if PREPROCESSED_SOURCE:
    restore_preprocessed_data(archive, DATA_DIR, DATA_NAME)
else:
    dataset_dir = extract_raw_dataset(archive, WORK_DIR / "raw_dataset")
    run_logged([sys.executable, "-u", "create_input_files.py",
                "--captions", str(dataset_dir / "LevirCCcaptions.json"),
                "--images", str(dataset_dir / "images"),
                "--output_folder", str(DATA_DIR), "--seed", str(DATA_SEED)], RUN_DIR / "data.log")
    if SAVE_PREPROCESSED:
        shutil.copy2(shutil.make_archive(str(WORK_DIR / "data_preprocessed"), "zip", DATA_DIR), RUN_DIR)
archive.unlink()

missing = [p.name for p in required_data_files(DATA_DIR, DATA_NAME) if not p.is_file()]
if missing:
    raise RuntimeError(f"Missing training files: {missing}")
preprocessing = DATA_DIR / "preprocessing_config.json"
update_run(RUN_DIR, data={
    "source": source, "archive_sha256": archive_sha256, "preprocessed_archive": bool(PREPROCESSED_SOURCE),
    "preprocessing": json.loads(preprocessing.read_text()) if preprocessing.is_file() else None,
})
print("Dataset archive SHA-256:", archive_sha256)

## 5. Train

Validation runs after every epoch. The latest and best checkpoints are uploaded to the run folder.

In [ ]:
command = [sys.executable, "-u", "train.py",
           "--data_folder", str(DATA_DIR), "--data_name", DATA_NAME,
           "--savepath", str(CHECKPOINT_DIR), "--drive_checkpoint_folder_id", DRIVE_CHECKPOINT_FOLDER_ID,
           "--epochs", str(TOTAL_EPOCHS), "--batch_size", str(BATCH_SIZE), "--workers", str(NUM_WORKERS),
           "--stop_criteria", str(STOP_CRITERIA), "--encoder_lr", str(ENCODER_LR),
           "--decoder_lr", str(DECODER_LR), *SEED_ARGS, *EXTRA_TRAIN_ARGS]
if HARMONIC_ALPHA:  # passed only when set, so the notebook also runs code versions without the option
    command += ["--harmonic_alpha", str(HARMONIC_ALPHA)]
if RESUME_CHECKPOINT:
    remote = Path(RESUME_CHECKPOINT)
    local = CHECKPOINT_DIR / "resume_source" / remote.name
    DriveCheckpoints(resolve_folder(service, remote.parent), service).download(remote.name, local)
    command += ["--checkpoint", str(local)]

try:
    run_logged(command, RUN_DIR / "train.log")
finally:
    metrics_path = CHECKPOINT_DIR / "training_metrics.jsonl"
    if metrics_path.is_file():
        shutil.copy2(metrics_path, RUN_DIR)
        update_run(RUN_DIR, training={"command": command, **training_summary(metrics_path)})

## 6. Evaluate on the test set

Also generates a caption for the example image pair.

In [ ]:
checkpoint = Path(EVAL_CHECKPOINT) if EVAL_CHECKPOINT else CHECKPOINT_DIR / ("BEST_" + CHECKPOINT_NAME)
if not checkpoint.is_file():
    raise FileNotFoundError(checkpoint)
results_dir = RUN_DIR / "eval_results" / timestamp()
shutil.rmtree(PROJECT_DIR / "eval_results_fortest" / "TEST", ignore_errors=True)

# eval.py and caption.py read every file in --path, so give them a folder with only this checkpoint.
with tempfile.TemporaryDirectory(dir="/content") as staging:
    os.symlink(checkpoint.resolve(), Path(staging) / checkpoint.name)
    data_args = ["--data_folder", str(DATA_DIR), "--data_name", DATA_NAME, "--path", staging, *SEED_ARGS]
    run_logged([sys.executable, "-u", "eval.py", "--Split", "TEST",
                "--beam_size", str(EVAL_BEAM_SIZE), *data_args], RUN_DIR / "eval.log")
    run_logged([sys.executable, "-u", "caption.py", "--img_A", "Example/A/train_000016.png",
                "--img_B", "Example/B/train_000016.png", *data_args], RUN_DIR / "eval.log")

shutil.copytree(PROJECT_DIR / "eval_results_fortest" / "TEST", results_dir)
shutil.copy2(PROJECT_DIR / "eval_results" / "resnet101_MCCFormers_diff_as_Q_trans_res.json",
             results_dir / "example_caption.json")
test_metrics = json.loads(next(results_dir.glob("*_scores.json")).read_text())
update_run(RUN_DIR, test={"checkpoint": str(checkpoint), "checkpoint_md5": checksum(checkpoint),
                          "beam_size": EVAL_BEAM_SIZE, "results": results_dir.name, "metrics": test_metrics})
print(json.dumps(test_metrics, indent=2))

## 7. Finish

Marks the run completed and, with `DISCONNECT_WHEN_DONE=True`, releases the runtime. Runtime-local files are lost; everything needed is in the run folder.

In [ ]:
update_run(RUN_DIR, status="completed", finished_utc=datetime.now(timezone.utc).isoformat(timespec="seconds"))
print("Run folder:", RUN_DIR)
if DISCONNECT_WHEN_DONE:
    from google.colab import runtime
    drive.flush_and_unmount()
    runtime.unassign()